In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from mamba_ssm import Mamba

/home/imvikranth/miniconda3/envs/mamba310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from collections import defaultdict

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [5]:
class VideoFrameDataset(Dataset):

    def __init__(self, root_dir, transform=None, frames_per_video=7):

        self.samples = []
        self.transform = transform

        classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {c:i for i,c in enumerate(classes)}

        for cls in classes:

            cls_path = os.path.join(root_dir, cls)
            videos = defaultdict(list)

            for img in os.listdir(cls_path):
                video_name = img.split("_f")[0]
                videos[video_name].append(img)

            for v in videos:

                frames = sorted(videos[v])[:frames_per_video]

                if len(frames) < frames_per_video:
                    continue

                paths = [os.path.join(cls_path,f) for f in frames]

                self.samples.append((paths, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        paths, label = self.samples[idx]

        frames = []

        for p in paths:

            img = Image.open(p).convert("RGB")

            if self.transform:
                img = self.transform(img)

            frames.append(img)

        frames = torch.stack(frames)

        return frames, label

In [6]:
train_dataset = VideoFrameDataset(
    root_dir="/home/imvikranth/mini_project/frame_dataset/train",
    transform=transform,
)

val_dataset = VideoFrameDataset(
    root_dir="/home/imvikranth/mini_project/frame_dataset/val",
    transform=transform,
)

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False ,
)

num_classes = len(train_dataset.class_to_idx)
print("Number of classes:", num_classes)
print(num_classes , type(num_classes))

Number of classes: 101
101 <class 'int'>


In [9]:
class MambaImageClassifier(nn.Module):

    def __init__(self, num_classes, d_model=128, patch_size=16):
        super().__init__()

        self.patch_embed = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)

        self.mamba = Mamba(
            d_model=d_model,
            d_state=16,
            d_conv=4,
            expand=2,
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(int(d_model),int(num_classes))

    def forward(self, x):   

        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.mamba(x)
        x = self.norm(x)
        x = x.mean(dim=1)

        return self.head(x)

In [10]:
frame_model = MambaImageClassifier(num_classes)
frame_model.load_state_dict(torch.load("mamba_image_model.pth"))
frame_model = frame_model.to(device)
frame_model.eval()

MambaImageClassifier(
  (patch_embed): Conv2d(3, 128, kernel_size=(16, 16), stride=(16, 16))
  (mamba): Mamba(
    (in_proj): Linear(in_features=128, out_features=512, bias=False)
    (conv1d): Conv1d(256, 256, kernel_size=(4,), stride=(1,), padding=(3,), groups=256)
    (act): SiLU()
    (x_proj): Linear(in_features=256, out_features=40, bias=False)
    (dt_proj): Linear(in_features=8, out_features=256, bias=True)
    (out_proj): Linear(in_features=256, out_features=128, bias=False)
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=128, out_features=101, bias=True)
)

In [11]:
class MambaImageEncoder(nn.Module):

    def __init__(self, frame_model):
        super().__init__()

        self.patch_embed = frame_model.patch_embed
        self.mamba = frame_model.mamba
        self.norm = frame_model.norm

    def forward(self, x):

        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1,2)

        x= x.contiguous()  

        x = self.mamba(x)
        x = self.norm(x)
        x = x.mean(dim=1)

        return x

In [12]:
encoder = MambaImageEncoder(frame_model).to(device)

for param in encoder.parameters():
    param.requires_grad = False 

<center>

### Temporal Mamba classifier
</center>

In [16]:
class TemporalMambaClassifier(nn.Module):

    def __init__(self, num_classes, d_model=128):
        super().__init__()

        self.encoder = encoder 

        self.temporal_mamba = Mamba(
            d_model=d_model,
            d_state=16,
            d_conv=4,
            expand=2
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):

        B,T,C,H,W = x.shape

        x = x.view(B*T, C, H, W)
        x = x.to(device)

        features = self.encoder(x)
        features = features.view(B, T, -1)
        features = features.contiguous()

        temporal_features = self.temporal_mamba(features)
        temporal_features = self.norm(temporal_features)

        video_feature = temporal_features.mean(dim=1)

        return self.head(video_feature)

In [17]:
model = TemporalMambaClassifier(num_classes).to(device)

criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.parameters(), lr=3e-4)

In [18]:
epochs = 20 

for epoch in range(epochs):
    model.train()
    total_loss = 0 

    for videos,labels in train_loader :
        videos = videos.to(device) 
        labels = labels.to(device) 

        optimizer.zero_grad() 
        outputs = model(videos) 

        loss = criterion(outputs,labels) 

        loss.backward() 
        optimizer.step()

        total_loss+= loss.item()
    print(f"Epoch{epoch+1},Loss:{total_loss/len(train_loader):.4f}")
    

Epoch1,Loss:2.5083
Epoch2,Loss:0.6885
Epoch3,Loss:0.2756
Epoch4,Loss:0.1332
Epoch5,Loss:0.0922
Epoch6,Loss:0.0565
Epoch7,Loss:0.0512
Epoch8,Loss:0.0526
Epoch9,Loss:0.0371
Epoch10,Loss:0.0383
Epoch11,Loss:0.0438
Epoch12,Loss:0.0277
Epoch13,Loss:0.0320
Epoch14,Loss:0.0346
Epoch15,Loss:0.0279
Epoch16,Loss:0.0223
Epoch17,Loss:0.0401
Epoch18,Loss:0.0266
Epoch19,Loss:0.0219
Epoch20,Loss:0.0313


Epoch1,Loss:2.5083
Epoch2,Loss:0.6885
Epoch3,Loss:0.2756
Epoch4,Loss:0.1332
Epoch5,Loss:0.0922
Epoch6,Loss:0.0565
Epoch7,Loss:0.0512
Epoch8,Loss:0.0526
Epoch9,Loss:0.0371
Epoch10,Loss:0.0383
Epoch11,Loss:0.0438
Epoch12,Loss:0.0277
Epoch13,Loss:0.0320
Epoch14,Loss:0.0346
Epoch15,Loss:0.0279
Epoch16,Loss:0.0223
Epoch17,Loss:0.0401
Epoch18,Loss:0.0266
Epoch19,Loss:0.0219
Epoch20,Loss:0.0313

In [19]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for videos, labels in val_loader:

        videos = videos.to(device)
        labels = labels.to(device)

        outputs = model(videos)

        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.8386132695756127


Total time : 41m 31.0s Validation Accuracy : 0.8386132695756127


In [20]:
torch.save(model.state_dict(), "temporal_mamba.pth")

<center>

### Temporal LSTM 
</center>

In [37]:
class TemporalLSTMClassifier(nn.Module):

    def __init__(self, encoder, num_classes, d_model=128, hidden_size=128, num_layers=1):
        super().__init__()

        self.encoder = encoder

        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.dropout = nn.Dropout(p=0.5)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)

        with torch.no_grad():
            features = self.encoder(x)          # (B*T, d_model)

        features = features.view(B, T, -1)      # (B, T, d_model)

        lstm_out, (h_n, c_n) = self.lstm(features)

        last_hidden = h_n[-1]                   # (B, hidden_size)
        out = self.dropout(last_hidden)
        out = self.norm(out)

        return self.head(out)

In [38]:
for param in encoder.parameters():
    param.requires_grad = True

model = TemporalLSTMClassifier(encoder, num_classes).to(device)

optimizer = optim.Adam([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.lstm.parameters(),    "lr": 1e-4},
    {"params": model.norm.parameters(),    "lr": 1e-4},
    {"params": model.head.parameters(),    "lr": 1e-4},
])

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=3, factor=0.5
)

In [39]:
best_acc = 0.0
epochs = 30

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for videos, labels in train_loader:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total * 100
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "temporal_lstm_best.pth")
        print(f"  --> Saved best model ({best_acc:.2f}%)")

print(f"\nBest Validation Accuracy: {best_acc:.2f}%")

Epoch  1 | Loss: 4.3445 | Val Acc: 27.38% | LR: 1.00e-05
  --> Saved best model (27.38%)
Epoch  2 | Loss: 3.6348 | Val Acc: 42.86% | LR: 1.00e-05
  --> Saved best model (42.86%)
Epoch  3 | Loss: 3.1490 | Val Acc: 51.70% | LR: 1.00e-05
  --> Saved best model (51.70%)
Epoch  4 | Loss: 2.7694 | Val Acc: 57.92% | LR: 1.00e-05
  --> Saved best model (57.92%)
Epoch  5 | Loss: 2.4617 | Val Acc: 63.06% | LR: 1.00e-05
  --> Saved best model (63.06%)
Epoch  6 | Loss: 2.2172 | Val Acc: 66.89% | LR: 1.00e-05
  --> Saved best model (66.89%)
Epoch  7 | Loss: 2.0284 | Val Acc: 69.10% | LR: 1.00e-05
  --> Saved best model (69.10%)
Epoch  8 | Loss: 1.8774 | Val Acc: 71.85% | LR: 1.00e-05
  --> Saved best model (71.85%)
Epoch  9 | Loss: 1.7564 | Val Acc: 73.64% | LR: 1.00e-05
  --> Saved best model (73.64%)
Epoch 10 | Loss: 1.6622 | Val Acc: 75.55% | LR: 1.00e-05
  --> Saved best model (75.55%)
Epoch 11 | Loss: 1.5744 | Val Acc: 76.81% | LR: 1.00e-05
  --> Saved best model (76.81%)
Epoch 12 | Loss: 1.51

Best Validation Accuracy: 83.86% Training time : 77m 9.5s


<center> 

### Temporal GRU 
</center> 

In [27]:
# Run this before training to confirm encoder is loaded correctly
print(next(encoder.parameters()).sum())  # should be a non-trivial number, not 0

tensor(-17.2885, device='cuda:0')


In [33]:
class TemporalGRUClassifier(nn.Module):

    def __init__(self, encoder, num_classes, d_model=128, hidden_size=128, num_layers=1):
        super().__init__()

        self.encoder = encoder

        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0       # dropout has no effect with num_layers=1, handled below
        )

        self.dropout = nn.Dropout(p=0.5)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)

        with torch.no_grad():
            features = self.encoder(x)          # (B*T, d_model)

        features = features.view(B, T, -1)      # (B, T, d_model)

        gru_out, h_n = self.gru(features)

        last_hidden = h_n[-1]                   # (B, hidden_size)
        out = self.dropout(last_hidden)         # dropout before classifier
        out = self.norm(out)

        return self.head(out)

In [34]:
# Unfreeze encoder
for param in encoder.parameters():
    param.requires_grad = True

model = TemporalGRUClassifier(encoder, num_classes).to(device)

# Use different learning rates — smaller for encoder, larger for GRU+head
optimizer = optim.Adam([
    {"params": model.encoder.parameters(), "lr": 1e-5},  # fine-tune slowly
    {"params": model.gru.parameters(),     "lr": 1e-4},
    {"params": model.norm.parameters(),    "lr": 1e-4},
    {"params": model.head.parameters(),    "lr": 1e-4},
])

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # label smoothing reduces overfit

In [36]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=3, factor=0.5
)

best_acc = 0.0
epochs = 20

for epoch in range(epochs):

    # --- Training ---
    model.train()
    total_loss = 0

    for videos, labels in train_loader:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping

        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total * 100
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "temporal_gru_best.pth")
        print(f"  --> Saved best model ({best_acc:.2f}%)")

print(f"\nBest Validation Accuracy: {best_acc:.2f}%")

Epoch  1 | Loss: 4.3145 | Val Acc: 26.42% | LR: 1.00e-05
  --> Saved best model (26.42%)
Epoch  2 | Loss: 3.5973 | Val Acc: 40.35% | LR: 1.00e-05
  --> Saved best model (40.35%)
Epoch  3 | Loss: 3.1405 | Val Acc: 50.27% | LR: 1.00e-05
  --> Saved best model (50.27%)
Epoch  4 | Loss: 2.7947 | Val Acc: 56.66% | LR: 1.00e-05
  --> Saved best model (56.66%)
Epoch  5 | Loss: 2.5142 | Val Acc: 61.21% | LR: 1.00e-05
  --> Saved best model (61.21%)
Epoch  6 | Loss: 2.3015 | Val Acc: 66.83% | LR: 1.00e-05
  --> Saved best model (66.83%)
Epoch  7 | Loss: 2.1285 | Val Acc: 68.44% | LR: 1.00e-05
  --> Saved best model (68.44%)
Epoch  8 | Loss: 1.9820 | Val Acc: 70.53% | LR: 1.00e-05
  --> Saved best model (70.53%)
Epoch  9 | Loss: 1.8682 | Val Acc: 72.03% | LR: 1.00e-05
  --> Saved best model (72.03%)
Epoch 10 | Loss: 1.7721 | Val Acc: 73.28% | LR: 1.00e-05
  --> Saved best model (73.28%)
Epoch 11 | Loss: 1.6948 | Val Acc: 75.07% | LR: 1.00e-05
  --> Saved best model (75.07%)
Epoch 12 | Loss: 1.63

In [32]:
# Check only GRU + head params are being trained, not encoder
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.encoder.parameters())
print(f"Trainable params: {trainable}")
print(f"Frozen encoder params: {frozen}")
# Trainable should only reflect GRU + norm + head, NOT the encoder

Trainable params: 717669
Frozen encoder params: 215168


Best Validation Accuracy: 81.35% Time taken : 50m 38s